# Xarray-Spatial Rasterize: Line rasterization with custom merge

Taxi GPS traces make a good stress test for line rasterization. 200,000
trajectories from Porto, each with a GPS fix every 15 seconds, give enough
density to trace the full street network. We use the built-in `sum` merge
alongside a custom log-sum merge to show how non-linear aggregation changes
what you can see.

### What you'll build

1. [Download and parse GPS trajectories](#Download-and-parse-GPS-trajectories) from the ECML/PKDD 2015 Porto taxi dataset
2. [Preview the raw trajectory data](#Preview)
3. [Rasterize trip counts](#Trip-count) with the built-in `count` merge
4. [Rasterize total duration](#Total-duration) with the built-in `sum` merge
5. [Write a custom log-sum merge function](#Custom-merge:-log-duration) that compresses dynamic range
6. [Compare all three side by side](#Comparison)

![Porto taxi rasterization preview](images/porto_taxi_lines_preview.png)

**Jump to a section:**
[Data](#Download-and-parse-GPS-trajectories) | [Preview](#Preview) | [Trip count](#Trip-count) | [Duration](#Total-duration) | [Custom merge](#Custom-merge:-log-duration) | [Comparison](#Comparison)

Standard imports plus `json` for parsing GPS polylines, `geopandas` and `shapely` for vector geometry, and `ngjit` for compiling the custom merge function.

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
import xarray as xr

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import LineString

import xrspatial  # registers .xrs accessor
from xrspatial.utils import ngjit

## Download and parse GPS trajectories

The [ECML/PKDD 2015 Taxi Trajectory Challenge](https://archive.ics.uci.edu/dataset/339/) dataset has 1.7 million taxi trips from Porto, Portugal. Each trip has a `POLYLINE` column with `[longitude, latitude]` pairs recorded every 15 seconds. We read the first 200,000 trips from the ~509 MB download.

In [ ]:
import urllib.request, zipfile, tempfile, os

url = ('https://archive.ics.uci.edu/static/public/339/'
       'taxi+service+trajectory+prediction+challenge+ecml+pkdd+2015.zip')

print('Downloading Porto taxi data (~509 MB)...')
tmpfile, _ = urllib.request.urlretrieve(url)

# Nested ZIP: outer contains train.csv.zip, which contains train.csv
tmpdir = tempfile.mkdtemp()
with zipfile.ZipFile(tmpfile) as outer:
    outer.extract('train.csv.zip', tmpdir)

with zipfile.ZipFile(os.path.join(tmpdir, 'train.csv.zip')) as inner:
    with inner.open('train.csv') as f:
        df = pd.read_csv(f, nrows=200_000,
                         usecols=['POLYLINE', 'MISSING_DATA'])

os.remove(tmpfile)

# Drop trips with missing GPS data
df = df[df.MISSING_DATA == False].copy()

# Parse polyline JSON, keep trips with at least 2 GPS points
df['coords'] = df.POLYLINE.apply(json.loads)
df = df[df.coords.apply(len) >= 2].copy()

# Trip duration in minutes (15 seconds between readings)
df['duration_min'] = df.coords.apply(lambda c: (len(c) - 1) * 0.25)

# Build LineStrings
df['geometry'] = df.coords.apply(LineString)
gdf = gpd.GeoDataFrame(df[['duration_min', 'geometry']],
                        geometry='geometry', crs='EPSG:4326')

print(f'{len(gdf):,} trajectories after filtering')
print(f'Duration: min={gdf.duration_min.min():.1f}, '
      f'median={gdf.duration_min.median():.1f}, '
      f'max={gdf.duration_min.max():.1f} minutes')

Each trajectory is a GPS trace through Porto's streets. Durations range from a few seconds to multi-hour shifts, with a median around 10 minutes.

## Preview

20,000 trajectories sampled and plotted on a dark background. The street network and major highways are already visible from the GPS traces alone.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
ax.set_facecolor('#1a1a2e')
gdf.sample(min(20_000, len(gdf)), random_state=42).plot(
    ax=ax, linewidth=0.1, alpha=0.3, color='#e94560')
ax.set_title(f'{len(gdf):,} taxi trajectories in Porto')
ax.set_axis_off()
ax.legend(handles=[Patch(facecolor='#e94560', alpha=0.6, label='GPS trace')],
          loc='lower right', fontsize=11, framealpha=0.9)
plt.tight_layout()

## Raster grid

A template DataArray covering Porto at roughly 25 meters per pixel. This grid sets the output resolution for all three rasterizations.

In [ ]:
bounds = (-8.72, 41.10, -8.52, 41.22)
width, height = 800, 600

def make_template(w, h, bounds):
    xmin, ymin, xmax, ymax = bounds
    px, py = (xmax - xmin) / w, (ymax - ymin) / h
    x = np.linspace(xmin + px / 2, xmax - px / 2, w)
    y = np.linspace(ymax - py / 2, ymin + py / 2, h)
    return xr.DataArray(np.zeros((h, w)), dims=['y', 'x'],
                        coords={'y': y, 'x': x})

template = make_template(width, height, bounds)
print(f'Grid: {width}x{height}, '
      f'~{111000 * (bounds[3] - bounds[1]) / height:.0f} m/pixel')

## Trip count

The simplest rasterization: count how many trajectories pass through each pixel. High-traffic roads light up. Side streets stay dark.

In [ ]:
count_raster = template.xrs.rasterize(gdf, merge='count', fill=0)

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.set_facecolor('black')
count_raster.where(count_raster > 0).plot.imshow(
    ax=ax, cmap='hot', add_colorbar=True,
    cbar_kwargs={'label': 'Trip count', 'shrink': 0.7})
ax.set_title('Trip count per pixel')
ax.set_axis_off()
plt.tight_layout()

## Total duration

Sum up trip durations with `merge='sum'` on the `duration_min` column. Each pixel accumulates the total taxi-minutes of all trips passing through it. Long trips add more weight, so the bright spots shift toward routes with both heavy traffic and longer rides.

In [ ]:
dur_raster = template.xrs.rasterize(
    gdf, column='duration_min', merge='sum', fill=0)

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.set_facecolor('black')
dur_raster.where(dur_raster > 0).plot.imshow(
    ax=ax, cmap='hot', add_colorbar=True,
    cbar_kwargs={'label': 'Total duration (minutes)', 'shrink': 0.7})
ax.set_title('Total duration per pixel (sum merge)')
ax.set_axis_off()
plt.tight_layout()

## Custom merge: log-duration

The built-in `sum` merge is linear: a 60-minute airport run contributes 12x more than a 5-minute hop. A few long trips can dominate a pixel's value.

A custom merge function fixes this. We define a `@ngjit`-compiled function that sums `log(1 + duration)` instead of raw duration. The log compresses the 12:1 ratio down to about 2:1 (`log(61) = 4.1` vs `log(6) = 1.8`).

The merge function receives three arguments:

- `pixel`: current pixel value
- `props`: 1D float64 array of feature properties (`props[0]` is the column value)
- `is_first`: 1 on first write to this pixel, 0 otherwise

In [ ]:
@ngjit
def log_duration_sum(pixel, props, is_first):
    """Sum log(1 + duration) instead of raw duration."""
    val = np.log1p(props[0])
    if is_first:
        return val
    return pixel + val

log_raster = template.xrs.rasterize(
    gdf, column='duration_min', merge=log_duration_sum, fill=0)

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.set_facecolor('black')
log_raster.where(log_raster > 0).plot.imshow(
    ax=ax, cmap='hot', add_colorbar=True,
    cbar_kwargs={'label': 'Log-duration sum', 'shrink': 0.7})
ax.set_title('Log-duration per pixel (custom merge)')
ax.set_axis_off()
plt.tight_layout()

## Comparison

All three rasterizations side by side. Count and duration look similar since busy roads carry both more trips and more total minutes. The log-duration panel looks different: it compresses the thousand-fold range between highways and side streets down to 3-4x, so the full street network becomes readable.

In [ ]:
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(20, 8), facecolor='black')

titles = ['Trip count', 'Total duration (sum)', 'Log-duration (custom merge)']
rasters = [count_raster, dur_raster, log_raster]

for ax, raster, title in zip(axes, rasters, titles):
    ax.set_facecolor('black')
    masked = raster.where(raster > 0)
    masked.plot.imshow(ax=ax, cmap='hot', add_colorbar=False,
                       interpolation='nearest')
    ax.set_title(title, color='white', fontsize=14, pad=10)
    ax.set_axis_off()

plt.tight_layout()
plt.savefig('images/porto_taxi_lines_preview.png',
            bbox_inches='tight', dpi=120, facecolor='black')

The count and duration panels are dominated by the main corridors and the downtown waterfront. Side streets barely register. The log-duration panel pulls those streets into visible range without blowing out the highways. Same data, different merge function.

When to reach for a non-linear merge: when a handful of features dominate the signal and you want to see the rest. Log-sum fits data that spans several orders of magnitude.

<div class="alert alert-block alert-warning">
<b>Coordinates are in WGS 84 (degrees).</b> The pixel grid here is defined in longitude/latitude, so pixels are not square on the ground. At Porto's latitude (~41°N), one degree of longitude covers about 84 km while one degree of latitude covers about 111 km. If metric pixel size matters for your analysis, reproject both the geometries and the grid to a local projected CRS (e.g. EPSG:3763 for Portugal) before rasterizing.
</div>

### References

- [ECML/PKDD 2015 Taxi Trajectory Challenge (UCI)](https://archive.ics.uci.edu/dataset/339/)
- [Bresenham's line algorithm (Wikipedia)](https://en.wikipedia.org/wiki/Bresenham%27s_line_algorithm)
- [xrspatial.rasterize API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.rasterize.html)
- [Custom merge functions (Notebook 28)](28_Rasterize.ipynb)